In [1]:
!pip install -q langchain langchain-community chromadb pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [2]:
import re
from langchain_community.document_loaders import PyPDFLoader
extract_path = '/kaggle/input/datasets/abedgaha/dataforrag/LIMITSFilesForRAG'
def clean_medical_text(text):
    # 1. إزالة الأسطر الفارغة المكررة والمسافات الزائدة
    text = re.sub(r'\s+', ' ', text)
    # 2. إزالة الرموز غير النصية (Non-ASCII) التي قد تنتج عن خطأ في قراءة الـ PDF
    text = re.sub(r'[^\x00-\x7f]', r'', text)
    # 3. إزالة أرقام الصفحات المتكررة (اختياري حسب تنسيق ملفاتك)
    text = re.sub(r'Page \d+', '', text)
    return text.strip()

# تطبيق التنظيف أثناء التحميل
def load_and_clean_docs(directory):
    cleaned_docs = []
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(directory, filename))
            pages = loader.load()
            for page in pages:
                page.page_content = clean_medical_text(page.page_content)
                cleaned_docs.append(page)
    return cleaned_docs

# استدعاء الدالة
all_documents = load_and_clean_docs(extract_path)
print(f"تم تحميل وتنظيف {len(all_documents)} صفحة طبية.")

NameError: name 'os' is not defined

In [ ]:
!pip install -q -U langchain langchain-community langchain-text-splitters chromadb pypdf sentence-transformers


In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# --- الخطوة 1: تقطيع النصوص (Text Splitting) ---
# سنستخدم final_chunks كمدخل للقاعدة
print("--- جاري تقطيع الوثائق المنظفة ---")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)

# تقسيم الوثائق التي نتجت من البلوك السابق (all_documents)
final_chunks = text_splitter.split_documents(all_documents)
print(f"تم إنشاء {len(final_chunks)} قطعة نصية.")



In [ ]:
print("--- تحميل نموذج الـ Embedding ---")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'} # استخدام GPU إذا كان متاحاً لتسريع العملية
)


In [ ]:
# --- الخطوة 3: إنشاء وحفظ ChromaDB ---
# تحديد مكان الحفظ في بيئة Kaggle
persist_directory = '/kaggle/working/medical_chroma_db'

print("--- جاري بناء قاعدة البيانات (قد يستغرق ذلك دقيقة) ---")
vector_db = Chroma.from_documents(
    documents=final_chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)

# حفظ القاعدة فعلياً على القرص
vector_db.persist()
print(f"تم بنجاح بناء وحفظ ChromaDB في: {persist_directory}")

In [ ]:
# لرؤية عدد العناصر المخزنة
print(f"عدد القطع النصية في القاعدة: {vector_db._collection.count()}")

# لجلب عينة من البيانات المخزنة (أول 3 عناصر مثلاً)
sample_data = vector_db.get(limit=3)
print("نص العينة المستخرج:")
print(sample_data['documents'])

In [ ]:
def run_retrieval_test(query, top_k=3):
    print(f"\n🔍 جاري البحث عن: '{query}'")
    print("="*50)
    
    # إجراء البحث بالتشابه (Similarity Search)
    # سيقوم الموديل بتحويل السؤال لمتجه تلقائياً والبحث في ChromaDB
    results = vector_db.similarity_search(query, k=top_k)
    
    if not results:
        print("❌ لم يتم العثور على نتائج. تأكد من أن الملفات تحتوي على معلومات ذات صلة.")
        return

    for i, doc in enumerate(results):
        # استخراج اسم الملف من الميتا-داتا
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        
        print(f"📄 النتيجة {i+1} | المصدر: {source}")
        print(f"📝 النص المسترجع: \n{doc.page_content}")
        print("-" * 50)

# --- تجربة النظام بأسئلة مختلفة ---

# الاختبار 1: سؤال مباشر عن الصوديوم
run_retrieval_test("What are the daily limits of sodium for hypertension?")

# الاختبار 2: سؤال عن الكوليسترول والقلب
run_retrieval_test("How does saturated fat affect heart disease?")